# 03 — Model Training
**FAILSAFE — XGBoost 3-Class Risk Model**

This notebook trains the exact same model used in the deployed website.

| Class | Label | G3 Score |
|-------|-------|----------|
| 0 | Low Risk | ≥ 14 |
| 1 | Medium Risk | 10–13 |
| 2 | High Risk | < 10 |

> ⚠️ **G3 is intentionally excluded from features** to prevent data leakage.

In [ ]:
!pip install xgboost shap -q
print('✅ Libraries ready!')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
import os

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder

print('✅ All imports successful!')

In [ ]:
# ── Load data ────────────────────────────────────────────────────────────────
df = pd.read_csv('../backend/processed_student_data.csv')

print('Shape:', df.shape)
print('Columns:', list(df.columns))
df.head(3)

In [ ]:
# ── Features used by the website (G3 excluded — data leakage) ───────────────
FEATURE_NAMES = [
    'G1', 'G2', 'absences', 'failures', 'studytime',
    'Medu', 'Fedu', 'goout', 'Dalc', 'Walc', 'health',
    'famrel', 'freetime', 'traveltime', 'age'
]

# Encode any remaining categorical columns
for col in df.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

# ── 3-class risk label from G3 ───────────────────────────────────────────────
def assign_risk(g3):
    if g3 >= 14:   return 0   # Low
    elif g3 >= 10: return 1   # Medium
    else:          return 2   # High

df['risk_label'] = df['G3'].apply(assign_risk)

print('Class distribution:')
print(df['risk_label'].value_counts().rename({0:'Low',1:'Medium',2:'High'}))

X = df[FEATURE_NAMES]
y = df['risk_label']

print(f'\nFeatures: {X.shape[1]}  |  Samples: {X.shape[0]}')

In [ ]:
# ── Train / test split ───────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f'Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}')

In [ ]:
# ── Train XGBoost (same hyperparameters as production) ───────────────────────
model = XGBClassifier(
    n_estimators=200,
    max_depth=4,            # Shallow — prevents overfitting
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,     # Prevents tiny-leaf memorisation
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    random_state=42,
    verbosity=0,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

print('✅ Model trained!')

In [ ]:
# ── Evaluation ───────────────────────────────────────────────────────────────
y_pred = model.predict(X_test)

print('=' * 55)
print('MODEL PERFORMANCE  (Target: 75–85% accuracy)')
print('=' * 55)
print(classification_report(
    y_test, y_pred,
    target_names=['Low Risk', 'Medium Risk', 'High Risk']
))

# Cross-validation
cv = cross_val_score(model, X, y, cv=5, scoring='f1_macro')
print(f'5-Fold CV F1 (macro): {cv.mean():.3f} ± {cv.std():.3f}')

In [ ]:
# ── Confusion matrix ─────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Low', 'Medium', 'High'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — FAILSAFE XGBoost')
plt.tight_layout()
plt.show()

In [ ]:
# ── SHAP Explainability ──────────────────────────────────────────────────────
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Summary plot for High Risk class (class 2)
plt.figure()
shap.summary_plot(
    shap_values[2], X_test,
    plot_type='bar',
    show=False
)
plt.title('Top Features — High Risk Prediction')
plt.tight_layout()
plt.show()

print('✅ SHAP explainer ready!')

In [ ]:
# ── Save model files (same paths as backend) ─────────────────────────────────
SAVE_DIR = '../backend'

joblib.dump(model,         os.path.join(SAVE_DIR, 'model.pkl'))
joblib.dump(explainer,     os.path.join(SAVE_DIR, 'explainer.pkl'))
joblib.dump(FEATURE_NAMES, os.path.join(SAVE_DIR, 'features.pkl'))

print('✅ Saved model.pkl, explainer.pkl, features.pkl → backend/')
print('\nDone! Commit the pkl files to deploy the updated model.')